### Notebook 2: Data Preprocessing & Cleaning
**Objective:** Clean non-predictive columns, handle missing data, inspect outliers, and export clean baseline data.

In [1]:
import os
import pandas as pd
import numpy as np

# Load raw dataset
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

# 1. Drop non-predictive customerID
df_clean = df.drop(columns=['customerID']).copy()

# 2. Convert TotalCharges to numeric (spaces become NaN)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'].str.strip(), errors='coerce')

# 3. Fill missing TotalCharges with 0.0 (tenure is 0)
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0.0)

print(f"Data cleaned. Current shape: {df_clean.shape}")
print(f"Missing values remaining: {df_clean.isnull().sum().sum()}")

Data cleaned. Current shape: (7043, 20)
Missing values remaining: 0


In [2]:
# 1. Encode Target Variable (Churn: Yes/No -> 1/0)
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

# 2. Identify categorical columns (excluding target)
cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

# 3. One-Hot Encode categorical features into 0s and 1s
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

print(f"Features converted! New dataset shape: {df_encoded.shape}")
df_encoded.head(3)

Features converted! New dataset shape: (7043, 31)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_31464\664708774.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True


In [3]:
# Check numerical features for extreme outliers using IQR
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in num_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_clean[(df_clean[col] < lower) | (df_clean[col] > upper)]
    print(f"{col:15s} | Outliers detected: {len(outliers)}")

tenure          | Outliers detected: 0
MonthlyCharges  | Outliers detected: 0
TotalCharges    | Outliers detected: 0


In [4]:
# Save fully encoded and cleaned dataset
os.makedirs("../data", exist_ok=True)
CLEAN_PATH = "../data/telco_cleaned.csv"
df_encoded.to_csv(CLEAN_PATH, index=False)

print(f"Converted dataset saved successfully to '{CLEAN_PATH}'!")

Converted dataset saved successfully to '../data/telco_cleaned.csv'!
